## Imports

In [1]:
import json
import random
from datetime import datetime, timezone
from hashlib import sha256

In [2]:
class Blockchain(object):
    def __init__(self, difficulty=4):
        self.chain = []
        self.pending_transactions = []
        self.difficulty = difficulty

        # Create the genesis block
        print("Creating genesis block")
        self.chain.append(self.mine_block())

    def add_transaction(self, sender, recipient, amount):
        """Queue a transaction to be included in the next mined block."""
        self.pending_transactions.append({
            "sender": sender,
            "recipient": recipient,
            "amount": amount,
        })

    def new_block(self, nonce):
        """Build a candidate block (not yet appended to the chain) for a given nonce."""
        block = {
            "index": len(self.chain),
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "transactions": self.pending_transactions,
            "previous_hash": self.last_block["hash"] if self.last_block else "0" * 64,
            "nonce": nonce,
        }
        # Get the hash of this new block, and add it to the block
        block["hash"] = self.hash(block)
        return block

    @staticmethod
    def hash(block):
        """
        Canonical hash of a block. Excludes the block's own 'hash' field (it's the
        output of this function, not an input -- including it would be circular).
        sort_keys + tight separators make the serialisation independent of dict
        construction order and free of incidental whitespace.
        """
        block_without_hash = {k: v for k, v in block.items() if k != "hash"}
        block_string = json.dumps(block_without_hash, sort_keys=True, separators=(",", ":")).encode()
        return sha256(block_string).hexdigest()

    @property
    def last_block(self):
        # Return the last block in the chain, or None if the chain is empty
        return self.chain[-1] if self.chain else None

    def valid_block(self, block):
        # Check if block's hash starts with `difficulty` leading hex zeros
        return block["hash"].startswith("0" * self.difficulty)

    def mine_block(self):
        """
        Proof-of-work: try nonce = 0, 1, 2, ... until the resulting hash satisfies
        valid_block(). Returns the mined block (not yet appended to the chain).
        """
        nonce = 0
        while True:
            candidate = self.new_block(nonce)
            if self.valid_block(candidate):
                self.pending_transactions = []  # reset only once mining succeeds
                return candidate
            nonce += 1

    def add_block(self):
        """Mine a block from the current pending transactions and append it."""
        block = self.mine_block()
        self.chain.append(block)
        return block

    def is_chain_valid(self):
        """
        Returns (bool, list[str]): overall validity + a list of problems found.
        Checks, per block:
          (a) stored hash matches a fresh recomputation (tamper evidence)
          (b) stored previous_hash matches the actual hash of the prior block
        """
        problems = []
        for i, block in enumerate(self.chain):
            if block["hash"] != self.hash(block):
                problems.append(
                    f"Block {i}: stored hash does not match recomputed hash "
                    f"(content was altered after mining)."
                )
            if i > 0:
                if block["previous_hash"] != self.chain[i - 1]["hash"]:
                    problems.append(
                        f"Block {i}: previous_hash does not match Block {i-1}'s "
                        f"current hash (chain link broken)."
                    )
        return (len(problems) == 0, problems)

## c) Create the chain (mines the genesis block automatically) and add block 1

Instantiating `Blockchain()` mines the genesis block in `__init__`. We then queue two
transactions and mine block 1 on top of it.

In [3]:
bc = Blockchain(difficulty=4)
genesis = bc.last_block
print(genesis)

Creating genesis block
{'index': 0, 'timestamp': '2026-08-12T12:13:30.355851+00:00', 'transactions': [], 'previous_hash': '0000000000000000000000000000000000000000000000000000000000000000', 'nonce': 388930, 'hash': '0000e3cddda4b35ed01d72914e6148ca1ad9ddd38562faeb914b8691aecc68cc'}


In [4]:
bc.add_transaction(sender="Sakhile", recipient="Thabo", amount=250.00)
bc.add_transaction(sender="Thabo", recipient="Lindiwe", amount=75.50)

block1 = bc.add_block()
print(block1)
print("\nprevious_hash matches genesis hash:", block1["previous_hash"] == genesis["hash"])

{'index': 1, 'timestamp': '2026-08-12T12:13:30.470877+00:00', 'transactions': [{'sender': 'Sakhile', 'recipient': 'Thabo', 'amount': 250.0}, {'sender': 'Thabo', 'recipient': 'Lindiwe', 'amount': 75.5}], 'previous_hash': '0000e3cddda4b35ed01d72914e6148ca1ad9ddd38562faeb914b8691aecc68cc', 'nonce': 6835, 'hash': '0000fb833fd5447ccc187be63e648e2af23ea95bef83981655c062e84a432005'}

previous_hash matches genesis hash: True


In [5]:
valid, problems = bc.is_chain_valid()
print("Chain valid?", valid)
for p in problems:
    print(" -", p)

Chain valid? True


## d) Tamper-evidence demonstration

Edit a past transaction inside block 1 **without** re-mining, and show the stored hash
no longer matches a fresh recomputation.

In [6]:
print("Original block 1 transactions:")
for tx in block1["transactions"]:
    print(" ", tx)

stored_hash_before = block1["hash"]
print("\nStored hash (unchanged field):", stored_hash_before)

print("\nAttacker edits a past transaction amount "
      "(250.00 -> 250000.00) without re-mining")
block1["transactions"][0]["amount"] = 250000.00

recomputed_hash_after = bc.hash(block1)
print("\nStored hash       :", block1["hash"])
print("Recomputed hash (from new data) :", recomputed_hash_after)
print("Do they match?                  :", block1["hash"] == recomputed_hash_after)

Original block 1 transactions:
  {'sender': 'Sakhile', 'recipient': 'Thabo', 'amount': 250.0}
  {'sender': 'Thabo', 'recipient': 'Lindiwe', 'amount': 75.5}

Stored hash (unchanged field): 0000fb833fd5447ccc187be63e648e2af23ea95bef83981655c062e84a432005

Attacker edits a past transaction amount (250.00 -> 250000.00) without re-mining

Stored hash       : 0000fb833fd5447ccc187be63e648e2af23ea95bef83981655c062e84a432005
Recomputed hash (from new data) : f67db35c733e833a15167f837150315572d1733f3299ec2b57bd7f8647e88cc2
Do they match?                  : False


In [7]:
valid, problems = bc.is_chain_valid()
print("Chain valid?", valid)
for p in problems:
    print(" -", p)

Chain valid? False
 - Block 1: stored hash does not match recomputed hash (content was altered after mining).


### Attempted cover-up: what if the attacker also re-mines block 1?

In [8]:
bc.add_transaction(sender="Lindiwe", recipient="Sakhile", amount=10.00)
block2 = bc.add_block()
print("Added block 2 on top of (already tampered) block 1.")
print("block2 previous_hash currently:", block2["previous_hash"])

# Attacker re-mines block 1 so its own hash matches the tampered transactions
bc.pending_transactions = []  # nothing pending; we're re-mining block 1 itself
nonce = 0
while True:
    candidate = {**block1}
    candidate["nonce"] = nonce
    candidate["hash"] = bc.hash({k: v for k, v in candidate.items() if k != "hash"} | {"nonce": nonce})
    if candidate["hash"].startswith("0" * bc.difficulty):
        break
    nonce += 1

print("\nAttacker re-mines block 1 with tampered data.")
print("Block 1 new hash              :", candidate["hash"])
print("Block 2 previous_hash         :", block2["previous_hash"])
print("Do they match?                :", candidate["hash"] == block2["previous_hash"])

# install the "fixed" block back into the chain to show the chain-level check still fails
bc.chain[1] = candidate

Added block 2 on top of (already tampered) block 1.
block2 previous_hash currently: 0000fb833fd5447ccc187be63e648e2af23ea95bef83981655c062e84a432005

Attacker re-mines block 1 with tampered data.
Block 1 new hash              : 0000570a8451f85bf2ec55e92f8c95bc748e8c21bb44c90e797c6e8da3460bea
Block 2 previous_hash         : 0000fb833fd5447ccc187be63e648e2af23ea95bef83981655c062e84a432005
Do they match?                : False


In [9]:
valid, problems = bc.is_chain_valid()
print("Chain valid after attempted cover-up?", valid)
for p in problems:
    print(" -", p)

Chain valid after attempted cover-up? False
 - Block 2: previous_hash does not match Block 1's current hash (chain link broken).
